# Week 6: Build a Defensible Classifier

**Challenge:** Classify participants as having elevated depression (PHQ-9 ≥ 5) vs minimal symptoms using the Boston College COVID-19 Sleep & Well-Being dataset.

**Models to compare:** Baseline → Logistic Regression → Decision Tree → Random Forest

**What makes this different from Week 4:** You're predicting a *category*, not a number. The type of mistake matters — missing someone with depression (false negative) is different from flagging someone who's fine (false positive).

---

**Cells 1–8 are done for you.** Run them to load and prepare the data. Your AI helps you fill in cells 9 onwards.

In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             precision_score, recall_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=1.1)

print('All imports loaded successfully!')

In [ ]:
# Cell 2: Load the three data files
data_dir = 'data/boston_college'

daily = pd.read_csv(f'{data_dir}/daily_survey.csv')
demo = pd.read_csv(f'{data_dir}/demographics.csv')
r1 = pd.read_csv(f'{data_dir}/round1_assessment.csv')

print(f'Daily survey: {daily.shape[0]:,} entries from {daily.sub_id.nunique():,} participants')
print(f'Demographics: {demo.shape[0]:,} participants')
print(f'Round 1:      {r1.shape[0]:,} entries from {r1.sub_id.nunique():,} unique participants')

In [ ]:
# Cell 3: Aggregate daily survey per participant
# Each participant has multiple daily entries — average them
agg_cols = ['PHQ9', 'PANAS_PA', 'PANAS_NA', 'TST', 'SE',
            'sleepdiary_sleeplatency', 'sleepdiary_wakes',
            'exercise', 'alcohol_bev', 'isolation', 'stress',
            'worry_scale', 'panas_sad3', 'panas_happy3', 'people_contact']

daily_with_phq = daily[daily['PHQ9'].notna()]
daily_agg = daily_with_phq.groupby('sub_id')[agg_cols].mean()

print(f'After aggregation: {daily_agg.shape[0]:,} participants with PHQ-9 data')

In [ ]:
# Cell 4: Score Big Five personality traits from BFI-2 items
r1_dedup = r1.drop_duplicates(subset='sub_id', keep='first').copy()
print(f'Round 1 after dedup: {len(r1_dedup)} (removed {len(r1) - len(r1_dedup)} duplicates)')

def score_trait(df, pos_items, neg_items):
    """Score a Big Five trait from positive and reverse-scored items."""
    pos = df[[f'big5_{i}' for i in pos_items]].values
    neg = 6 - df[[f'big5_{i}' for i in neg_items]].values
    all_items = np.concatenate([pos, neg], axis=1)
    return np.nanmean(all_items, axis=1)

r1_dedup['Extraversion'] = score_trait(r1_dedup, [1, 11, 21], [6, 16, 26])
r1_dedup['Agreeableness'] = score_trait(r1_dedup, [7, 17, 27], [2, 12, 22])
r1_dedup['Conscientiousness'] = score_trait(r1_dedup, [3, 13, 23], [8, 18, 28])
r1_dedup['Neuroticism'] = score_trait(r1_dedup, [4, 14, 24], [9, 19, 29])
r1_dedup['Openness'] = score_trait(r1_dedup, [5, 15, 25], [10, 20, 30])

# GAD-7 total
gad_cols = [f'gad_{i}' for i in range(1, 8)]
r1_dedup['GAD7_total'] = r1_dedup[gad_cols].sum(axis=1)

print('Big Five traits and GAD-7 scored successfully!')

In [ ]:
# Cell 5: Merge all three dataframes
merged = daily_agg.merge(
    demo[['sub_id', 'age1', 'bio_sex']],
    left_index=True, right_on='sub_id', how='inner'
)
print(f'After merge with demographics: {merged.shape}')

merged = merged.merge(
    r1_dedup[['sub_id', 'Extraversion', 'Agreeableness', 'Conscientiousness',
              'Neuroticism', 'Openness', 'GAD7_total']],
    on='sub_id', how='inner'
)
merged = merged.drop_duplicates(subset='sub_id')
print(f'After merge with Round 1: {merged.shape}')

In [ ]:
# Cell 6: Create binary target
# PHQ-9 >= 5 = at least mild depression ("elevated")
# PHQ-9 < 5 = minimal symptoms
merged['depression_elevated'] = (merged['PHQ9'] >= 5).astype(int)

print('Class balance:')
print(merged['depression_elevated'].value_counts())
print()
print('Proportions:')
print(merged['depression_elevated'].value_counts(normalize=True).round(3))

In [ ]:
# Cell 7: Define features and target
feature_cols = ['PANAS_PA', 'PANAS_NA', 'TST', 'SE',
                'sleepdiary_sleeplatency', 'sleepdiary_wakes',
                'exercise', 'alcohol_bev', 'isolation', 'stress',
                'worry_scale', 'panas_sad3', 'panas_happy3',
                'age1', 'bio_sex',
                'Extraversion', 'Agreeableness', 'Conscientiousness',
                'Neuroticism', 'Openness', 'GAD7_total']

X = merged[feature_cols].copy()
y = merged['depression_elevated'].copy()

# Handle missing values (a few Big Five scores are missing)
imputer = SimpleImputer(strategy='median')
X = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols, index=X.index)

print(f'Features: {len(feature_cols)} columns')
print(f'Samples: {len(X)}')

## Summary so far

You now have:
- `X` — a DataFrame with 21 features for ~836 participants
- `y` — a binary target (1 = elevated depression, 0 = minimal)
- Class balance is roughly 54.5% elevated / 45.5% minimal

**From here, your AI helps you build the classification pipeline.** The cells below give you the structure — ask your AI to fill in the code.

---

## Step 1: Train/Test Split

Split the data 80/20 with stratification (keeps class proportions equal in both sets).

Ask your AI: *"How do I do a stratified train/test split with sklearn?"*

In [ ]:
# Cell 9: Train/test split
# YOUR CODE HERE — ask your AI to fill this in
# X_train, X_test, y_train, y_test = train_test_split(...)

## Step 2: Baseline Model

The majority-class baseline always predicts the most common class. Any real model must beat this.

Ask your AI: *"How do I create a majority-class baseline with DummyClassifier and report accuracy and F1?"*

In [ ]:
# Cell 10: Baseline model
# YOUR CODE HERE

## Step 3: Logistic Regression

The simplest real classifier. Remember to scale your features first (logistic regression is sensitive to feature scales).

Ask your AI: *"How do I fit a LogisticRegression with StandardScaler and report accuracy, F1, AUC, and the confusion matrix?"*

In [ ]:
# Cell 11: Logistic Regression
# YOUR CODE HERE

## Step 4: Decision Tree

Use `max_depth=5` to prevent overfitting. Trees don't need feature scaling.

Ask your AI: *"How do I fit a DecisionTreeClassifier with max_depth=5 and report the same metrics?"*

In [ ]:
# Cell 12: Decision Tree
# YOUR CODE HERE

## Step 5: Random Forest

100 trees, `max_depth=10`. The ensemble should reduce variance compared to a single tree.

Ask your AI: *"How do I fit a RandomForestClassifier with 100 trees and report metrics including AUC?"*

In [ ]:
# Cell 13: Random Forest
# YOUR CODE HERE

## Step 6: Model Comparison

Create a bar chart or table comparing all four models on accuracy, F1, and AUC.

Ask your AI: *"Create a grouped bar chart comparing my four models on accuracy, F1, and AUC."*

In [ ]:
# Cell 14: Model comparison chart
# YOUR CODE HERE

## Step 7: Feature Importance

Which features matter most for predicting depression? Use Random Forest feature importances.

Ask your AI: *"Plot the top 10 features by Random Forest importance as a horizontal bar chart."*

In [ ]:
# Cell 15: Feature importance
# YOUR CODE HERE

## Step 8: Threshold Analysis

The default threshold is 0.5, but you should try others. For a depression screener, you might lower it to catch more cases.

Ask your AI: *"Try thresholds 0.3, 0.4, 0.5, 0.6, 0.7 on my best model and show how precision, recall, and F1 change."*

In [ ]:
# Cell 16: Threshold analysis
# YOUR CODE HERE

## Step 9: Confusion Matrices

Show confusion matrices for your best model, ideally at your chosen threshold.

Ask your AI: *"Plot confusion matrices side by side for threshold 0.3 and 0.5."*

In [ ]:
# Cell 17: Confusion matrices
# YOUR CODE HERE

## Step 10: Ethical Consideration

In the cell below, write 2–3 sentences about one ethical concern if this model were deployed to make real decisions about people. Think about: Who might be harmed? Which errors are worse? Does the model work equally well for everyone?

*Your ethical consideration here...*

## Step 11: Save Your Results

Save any figures you want to include in your presentation.

In [ ]:
# Cell 18: Save figures
# Uncomment when ready:
# fig.savefig('model_comparison.png', dpi=300, bbox_inches='tight')
# print('Figure saved!')